In [ ]:
# Standard packages

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

# Model

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_sample_weight

# Pipeline + Transformer:

from sklearn.preprocessing import OneHotEncoder, StandardScaler,OrdinalEncoder, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Neural network

import torch
from torch import nn
import cudf

# Speed up options
%load_ext cudf.pandas
%load_ext cuml.accel



In [ ]:
# Reading in data

train = pd.read_csv('train.csv')

test = pd.read_csv('test.csv')

orig = pd.read_csv('f1_strategy_dataset_v4.csv')

In [ ]:
# Setting up target columns and test IDs

ytrain = train['PitNextLap']

yorig = orig['PitNextLap']

test_id = test['id']

In [ ]:
# Removing ID column and target

Xtrain = train.drop(columns=['id','PitNextLap'], axis = 1).copy()

Xtest = test.drop(columns = ['id'], axis = 1).copy()

Xorig = orig.drop(columns= ['Normalized_TyreLife', 'PitNextLap'], axis = 1).copy()

# Data cleaning #

In [ ]:


# Null values

Xtrain.info(), Xtest.info() , Xorig.info() # No null values


In [ ]:
num_cols = Xtrain.select_dtypes(['int','float']).columns.to_list()
num_cols

In [ ]:
# Understanding if any feature outliers have a significant impact on target

sns.displot(
    data=train,
    x = 'TyreLife',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd'
)

sns.displot(
    data=train,
    x = 'Stint',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd',
    discrete = True
)

sns.displot(
    data=train,
    x = 'LapNumber',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd',
    discrete = True
)


sns.displot(
    data=train,
    x = 'Position',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd',
    discrete = True
)

sns.displot(
    data=train,
    x = 'LapTime (s)',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd'
)

sns.displot(
    data=train,
    x = 'LapTime_Delta',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd'
)


sns.displot(
    data=train,
    x = 'Cumulative_Degradation',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd',
    discrete = True
)

sns.displot(
    data=train,
    x = 'RaceProgress',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd',
    discrete = True
)

sns.displot(
    data=train,
    x = 'Position_Change',
    col = 'PitNextLap',
    height = 5,
    bins = 'fd',
    discrete = True
)


In [ ]:
# Summary statisitcs
Xtrain[num_cols].describe()


In [ ]:
# Winsorization
cols_to_remove = ['PitStop','PitNextLap'] # Binary columns that contain imbalanced responses

def num_outliers(df, choice):

    """
    The function applies the moethod of winsorization to outliers. Choice is a boolean repsonse. True applies the method to the data
    frame. Fasle just prints out any information you would like to know about the data frame 
    """

    df_new = df.copy()

    num_cols = df_new.select_dtypes(['int','float']).columns.to_list()

    num_cols = list(set(num_cols) - set(cols_to_remove))

    df_new[num_cols] = df_new[num_cols].astype('float')

    for col in num_cols:

        q1 = np.quantile(df_new[col], 0.25)

        q3 = np.quantile(df_new[col], 0.75)

        iqr = q3 - q1

        lower = q1 - 1.5 * iqr

        upper = q3 + 1.5 * iqr

        # Mask returns true/falses where data column meets outlier criteria

        mask = (df_new[col] < lower) | (df_new[col] > upper)

        low_mask = (df_new[col] < lower)

        high_mask = (df_new[col] > upper)

        n_outliers = mask.sum()

        print(f'{col} has {n_outliers} outliers')

        if choice == True: # Winsorization 

            df_no_outliers = df_new.loc[~mask]

            df_out_low = df_new.loc[low_mask]

            df_out_high = df_new.loc[high_mask]

            df_out_high.loc[:, col] = upper

            df_out_low.loc[:, col] = lower

            df_new = pd.concat([df_no_outliers, df_out_high, df_out_low], axis = 0).sort_index()
        else:
            print('No changes made to dataframe')



    print(f'Shape of new data frame {df_new.shape} | Shape of old data frame {df.shape}')


    return df_new



In [ ]:
Xtrain_outliers = num_outliers(Xtrain, True)
print('//////////')
Xtest_outliers = num_outliers(Xtest, True)
print('//////////')
Xorig_outliers = num_outliers(Xorig, True)

In [ ]:
# Seeing how the distribution of the numerical columns changes after function is applied
Xtrain_outliers[num_cols].describe()

In [ ]:
cat_cols = Xtrain_outliers.select_dtypes('object').columns.to_list()
cat_cols

In [ ]:
# Identifying any potential columns with categorical outliers or anomalies, mis-entries etc. and what columns to encode
Xtrain_outliers[cat_cols].describe()

In [ ]:
Xtrain_outliers['Driver'].unique()

In [ ]:
Xtrain_outliers['Driver'].str.startswith('D1')

In [ ]:
# Categorical outliers

def cat_outliers(df):

    df_new = df.copy()

    for i in range(10):

        df_new['Driver'] = df_new['Driver'].apply(lambda x: f'D{i}00s' if x.startswith(f'D{i}') else x)


    print('All categorical outliers were removed')

    return df_new




In [ ]:
Xtrain_cat_outliers = cat_outliers(Xtrain_outliers)
Xtest_cat_outliers = cat_outliers(Xtest_outliers)
Xorig_cat_outliers = cat_outliers(Xorig_outliers)

# Feature engineering #

In [ ]:
# Categorical feature engineering

def cat_fe(df):

    df_new = df.copy()

    for i, c1 in enumerate(cat_cols):
        for j, c2 in enumerate(cat_cols):
            if i < j:

                df_new[f'BI_Combin{c1}_{c2}'] = df_new[c1].astype(str) + "_" + df_new[c2].astype(str)

    print('Categorical Feature Engineering complete')

    return df_new


In [ ]:
Xtrain_cat = cat_fe(Xtrain_cat_outliers)

Xtest_cat = cat_fe(Xtest_cat_outliers)

Xorig_cat = cat_fe(Xorig_cat_outliers)

In [ ]:
# Numerical feature engineering

def num_fe(df):

    df_new = df.copy()

    df_new['Lap_x_Delta'] = df_new['LapTime (s)'] * df_new['LapTime_Delta']

    df_new['Average_Position_Change'] = (df_new['Position'] + df_new['Position_Change'])/2

    df_new['TyreLife_Left'] = df_new['TyreLife']/(df_new['LapNumber'])

    print('Numerical Feature Engineering is complete')


    return df_new



In [ ]:
Xtrain_num = num_fe(Xtrain_cat)

Xtest_num = num_fe(Xtest_cat)

Xorig_num = num_fe(Xorig_cat)

In [ ]:
Xorig_num['Race'].unique()

In [ ]:
# Removing rows containing races not included in test or training set

mask = (Xorig_num['Race'] == 'Pre-Season Test') | (Xorig_num['Race'] == 'Pre-Season Track Session' )

Xorig_num = Xorig_num[~mask].reset_index(drop=True)

In [ ]:
# Frequency encoding
def freq_encoder(df):

    df_new = df.copy()

    cat_cols = df_new.select_dtypes('object').columns.to_list()

    freq_cols = list(set(cat_cols) - set(['Compound']))

    for col in freq_cols:

        freq_tab = pd.concat([Xtrain_num[col], Xtest_num[col], Xorig_num[col]]).value_counts()

        df_new[f'freq_{col}'] = df_new[col].map(freq_tab).fillna(0).astype(float)

    print('Frequency encoding is complete')

    df_new = df_new.drop(columns = [], axis = 1)

    return df_new



In [ ]:
Xtrain_freq = freq_encoder(Xtrain_num)

Xtest_freq = freq_encoder(Xtest_num)

Xorig_freq = freq_encoder(Xorig_num)

In [ ]:


num_pro = Xtrain_freq.select_dtypes(['int','float']).columns.to_list()

cat_pro = ['Compound']



In [ ]:
len(num_pro),len(cat_pro),Xtrain_freq.shape

# Preprocessing #

In [ ]:
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]), num_pro),

    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(drop='first', sparse_output=False))
    ]), cat_pro)
])

def data_prep(df):

    df_new = df.copy()

    ImputedX = preprocessor.fit_transform(Xtrain_freq)

    colnames = preprocessor.get_feature_names_out()

    df_imputed = preprocessor.transform(df_new)

    df_final = pd.DataFrame(df_imputed, columns=colnames, index=df.index)

    print('Data preprocessing is complete')

    return df_final

In [ ]:
Xtrain_pre = data_prep(Xtrain_freq)

Xtest_pre = data_prep(Xtest_freq)

Xorig_pre = data_prep(Xorig_freq)

In [ ]:
Xtest_pre.columns

# Neural network #

In [ ]:

num_pre = Xtrain_pre.select_dtypes(['int','float']).columns.to_list()

cat_pre  = Xtrain_pre.select_dtypes('object').columns.to_list()
te_features = cat_pre

In [ ]:
# Device agnostics

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

epochs = 1000

for fold, (train_idx, val_idx) in enumerate(skf.split(Xtrain_pre, ytrain)):
    print(f'Training fold_{fold}')

    # Train/Test spli
    X_train, X_val = Xtrain_pre.iloc[train_idx], Xtrain_pre.iloc[val_idx]

    y_train, y_val  = ytrain.iloc[train_idx], ytrain.iloc[val_idx]
      

    # Intialising model
    model_nn = nn.Sequential(
        nn.Linear(X_train.shape[1], 100),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(100, 50),
        nn.ReLU(),
        nn.Linear(50, 10),
        nn.ReLU(),
        nn.Linear(10, 1)
    ).to(device) # Transfer to gpu if available

    # Loss & optimizer fuctions
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.SGD(model_nn.parameters(), lr=0.1)

    # Convert to dataframes to tensors
    X_train_t = torch.tensor(X_train.values, dtype=torch.float32).to(device)
    X_val_t   = torch.tensor(X_val.values, dtype=torch.float32).to(device)

    y_train_t = torch.tensor(y_train.values, dtype=torch.float32).to(device)
    y_val_t   = torch.tensor(y_val.values, dtype=torch.float32).to(device)

    for epoch in range(epochs):

        # Train
        model_nn.train()

        # Forward pass
        y_logits = model_nn(X_train_t).squeeze()
        y_probs  = torch.sigmoid(y_logits)

        # Loss function
        loss = loss_fn(y_logits, y_train_t)

        # Optimzier zero grad
        optimizer.zero_grad()

        # Backward propagation
        loss.backward()

        # Optimizer step
        optimizer.step()

        # Evaluation metric for competition 
        train_auc = roc_auc_score(
            y_train_t.detach().cpu().numpy(), # Converts data back into numpy arrays to work with sklearn 
            y_probs.detach().cpu().numpy()
        )

        # Test
        model_nn.eval()
        with torch.inference_mode():

            # Forward pass
            val_logits = model_nn(X_val_t).squeeze()
            val_probs  = torch.sigmoid(val_logits)

            # Loss 
            val_loss = loss_fn(val_logits, y_val_t)

            # Evaluation metric 
            val_auc = roc_auc_score(
                y_val_t.cpu().numpy(),
                val_probs.cpu().numpy()
            )

        # Logging
        if epoch % 100 == 0:
            print(f'Fold {fold} | Epoch {epoch} | '
                  f'Train Loss: {loss:.5f}, AUC: {train_auc:.4f} | '
                  f'Val Loss: {val_loss:.5f}, AUC: {val_auc:.4f}')